# Agri-JAX tutorial: the DSSAT-CSM maize model in JAX

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juksentang/agri-jax/blob/main/tutorial/agrijax_tutorial.ipynb) · [中文版](https://colab.research.google.com/github/juksentang/agri-jax/blob/main/tutorial/agrijax_tutorial_zh.ipynb)

Agri-JAX re-implements the DSSAT-CSM v4.8.6 maize model (soil water, evapotranspiration, root water uptake, CERES-Maize) from its published equations in JAX, and reads DSSAT's own experiment files; we use the original DSSAT program only as the reference to check against. It is an independent implementation from published equations, validated against DSSAT-CSM (BSD-3) and RZWQM2 outputs. DSSAT-CSM, its data and the example experiment are the work of the DSSAT Foundation and its contributors (BSD-3); this tutorial stands on that work.

The example is DSSAT's experiment **UFGA8201** (Gainesville, Florida, 1982), treatment 4 (irrigated, high nitrogen); the model runs **nitrogen off**.

1. **Run Agri-JAX**: install, read the experiment, simulate one season. No DSSAT program is involved.
2. **Compare with DSSAT**: fetch the original DSSAT program (the validated v4.8.6.0 build) and overlay its run.
3. **Batches and speed**: weather years × sowing dates × cultivar samples in one batched call, timed on this machine, and DSSAT on the same machine as the reference.
4. **Calibration**: the cultivar coefficients fitted on treatment 4, treatment 6 held out, without DSSAT.
5. **Round trip into DSSAT**: the calibrated `.CUL` row run in DSSAT.

*Runtime → Run all* should take a few minutes on a free CPU runtime. Every time printed below is measured in this session, on the machine printed with it.

## 1. Run Agri-JAX

On Colab the next cell installs Agri-JAX from GitHub (the PyPI package is only a placeholder for now) with its plotting extra; elsewhere it does nothing (install with `pip install "agrijax[plot] @ git+https://github.com/juksentang/agri-jax"`). If Colab asks to restart the session after the install, restart and continue from the cell after it.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q "agrijax[plot] @ git+https://github.com/juksentang/agri-jax"
LANG = "en"  # the language of this notebook's messages ("en" or "zh")

In [ ]:
# Printed messages of this notebook in English and Chinese (tutorial/source/text.toml)
MSG = {
    "machine": {
        "en": "machine: {machine}",
        "zh": "测试机器：{machine}",
    },
    "season": {
        "en": "Agri-JAX, {treatment}: silking {adat}, maturity {mdat}, grain yield {hwam:.0f} kg/ha",
        "zh": "Agri-JAX，{treatment}：吐丝期 {adat}，成熟期 {mdat}，籽粒产量 {hwam:.0f} kg/ha",
    },
    "scenarios": {
        "en": "{n} scenarios built from DSSAT's files (no DSSAT run)",
        "zh": "基于 DSSAT 文件构建了 {n} 个情景（未运行 DSSAT）",
    },
    "check_yield": {
        "en": "published cultivar on {n} scenarios: Agri-JAX yield within {rel:.3%} of DSSAT",
        "zh": "已发表品种在 {n} 个情景下：Agri-JAX 产量与 DSSAT 的相对误差在 {rel:.3%} 以内",
    },
    "check_dates": {
        "en": "silking date equal on {adat} and maturity date on {mdat} of {n} scenarios",
        "zh": "{n} 个情景中：吐丝期一致数 {adat}，成熟期一致数 {mdat}",
    },
    "dssat_per_scenario": {
        "en": "DSSAT took {sec:.2f} s per scenario on this machine (one process each)",
        "zh": "DSSAT 在当前机器上平均每情景耗时 {sec:.2f} s（每个情景单独立进程）",
    },
    "batch_yield": {
        "en": "{n} cultivars, 1982: Agri-JAX yield within {rel:.3%} of DSSAT",
        "zh": "1982 季，{n} 个品种：Agri-JAX 产量与 DSSAT 的相对误差在 {rel:.3%} 以内",
    },
    "batch_dates": {
        "en": "maturity date equal on {same} of {n}",
        "zh": "{n} 个品种中成熟期一致数：{same}",
    },
    "dssat_speed": {
        "en": "DSSAT:    {ms:.2f} ms per season (measured: {n} seasons in {s:.2f} s, one batch run)",
        "zh": "DSSAT:    每季 {ms:.2f} ms（实测：单次批处理运行 {s:.2f} s 完成 {n} 个生长季）",
    },
    "ours_speed": {
        "en": "Agri-JAX: {ms:.2f} ms per season (measured: {n} seasons in {s:.2f} s)",
        "zh": "Agri-JAX: 每季 {ms:.2f} ms（实测：{s:.2f} s 完成 {n} 个生长季）",
    },
    "compile": {
        "en": "Agri-JAX compiled its batched program once, in {s:.1f} s (first call)",
        "zh": "Agri-JAX 批处理程序单次编译耗时：{s:.1f} s（首次调用）",
    },
    "extrapolated": {
        "en": "{n} seasons: DSSAT about {s:.0f} s (EXTRAPOLATED from its measured per-season time)",
        "zh": "{n} 个生长季：DSSAT 预估耗时约 {s:.0f} s（根据单季实测耗时外推）",
    },
}


def say(key, **values):
    """Print message ``key`` in the language of this notebook."""
    print(MSG[key][LANG].format(**values))

In [ ]:
import os

import numpy as np

import agrijax as aj

SMALL = os.environ.get("AGRI_JAX_TUTORIAL_SMALL") == "1"  # a quick run, used by the tests
say("machine", machine=aj.machine())

`aj.dssat.experiment("UFGA8201")` reads DSSAT's example experiment. Its files are DSSAT's public data, downloaded once from DSSAT's own repositories at fixed versions and checked against their SHA-256: the FileX, the observations, the Gainesville weather of 1978–1987 and the soil file from `DSSAT/dssat-csm-data`, the maize cultivar, ecotype and species files and the standard data from `DSSAT/dssat-csm-os` v4.8.6.0 (BSD-3).

The table lists the treatments with the cultivar, whether Agri-JAX can build the treatment's inputs, and how much DSSAT's own yield changes when nitrogen is simulated. The model here runs nitrogen off, so the treatments where nitrogen matters (1, 3, 5, the low-nitrogen ones, and 2) are not used for fitting; treatments 4 and 6 are.

In [ ]:
exp = aj.dssat.experiment("UFGA8201")
print(exp)
exp.table()

`exp.run(treatment=4)` simulates the season with Agri-JAX alone. The inputs are built from the experiment's files by Agri-JAX's implementation of DSSAT's input chain: the soil layers and initial water from the FileX and `.SOL`, the daily weather from the `.WTH`, the irrigation schedule, the cultivar, ecotype and species coefficients from the `.CUL` / `.ECO` / `.SPE`; the season runs from the simulation start to the crop's own maturity. The first run also compiles the model; that time is part of the cell's run time. The figure shows leaf area index, tops and grain weight and the soil water of the profile, with the observations of the experiment's T file (circles).

In [ ]:
season = exp.run(treatment=4)
s = season.summary
say("season", treatment=season.treatment, adat=int(s["ADAT"]), mdat=int(s["MDAT"]), hwam=s["HWAM"])
aj.plot.season(season, observed=exp.observed(4));

In [ ]:
season.to_frame().tail()

## 2. Compare with DSSAT

`aj.dssat.install_reference()` fetches the original DSSAT program, `dscsm048` v4.8.6.0: the prebuilt, statically linked build that all of Agri-JAX's validation runs use (a release asset of the Agri-JAX repository, checked against its SHA-256; it carries DSSAT's licence and a provenance note). `exp.reference(treatment=4)` runs it on the same treatment, nitrogen off.

The end-of-season values: silking (`ADAT`) and maturity (`MDAT`) dates, the difference in days; grain yield (`HWAM`) and tops weight (`CWAM`) at maturity, kg ha⁻¹, the relative difference.

In [ ]:
aj.dssat.install_reference()
ref = exp.reference(treatment=4)
season.compare_summary(ref)

Day by day: leaf area index, tops and grain weight (`PlantGro.OUT`) and the soil water of the profile (`SoilWat.OUT`), on every day both report. DSSAT prints these values rounded (LAI to 0.01, weights to 1 kg ha⁻¹, soil water to 1 mm), which bounds the differences one can see. The figures overlay DSSAT's run (dashed) and plot Agri-JAX against DSSAT.

In [ ]:
season.compare_daily(ref)

In [ ]:
aj.plot.season(season, reference=ref, observed=exp.observed(4));

In [ ]:
aj.plot.compare(season, ref);

## 3. Batches and speed

The same treatment on other weather years and sowing dates: `exp.scenarios` moves the whole management calendar of the FileX (simulation start, planting, irrigation, fertilizer dates) to each weather year and by each sowing shift and builds each scenario's inputs, without running DSSAT. These scenarios are not part of the validation set, so they are checked against DSSAT below. They are nitrogen-off simulations of one cultivar at one site: an illustration of the workflow, not a recommendation for sowing dates.

In [ ]:
years = range(1981, 1984) if SMALL else range(1978, 1988)
shifts = [0] if SMALL else [-14, 0, 14]
scen = exp.scenarios(treatment=4, years=years, sowing_shift=shifts)
say("scenarios", n=len(scen.runs))
scen.table

`scen.run(cultivar)` simulates every cultivar sample on every scenario in one batched program: the model's `scan` over the days inside a `vmap` over the seasons, compiled once. Its `timing` separates the compilation (`compile_s`, 0 when the program was compiled before) from the run (`run_s`, with the results back on the host). The second call reuses the compiled program with new samples: its time is the steady-state cost. The figure shows the yield distribution of all samples per weather year.

In [ ]:
K = 8 if SMALL else 320
pub = scen.published


def samples(seed):
    """K cultivars within +-10 % of the published one; sample 0 is the published cultivar."""
    rng = np.random.default_rng(seed)
    c = {n: pub[n] * rng.uniform(0.9, 1.1, K) for n in ("P1", "P5", "G2", "G3", "PHINT")}
    for n in c:
        c[n][0] = pub[n]
    return c


first = scen.run(samples(0))
first

In [ ]:
second = scen.run(samples(1))
second

In [ ]:
aj.plot.batch(first, by="year");

`scen.reference()` runs DSSAT on every scenario at the published cultivar (one process per scenario). Sample 0 of each scenario is the published cultivar, so Agri-JAX and DSSAT must agree on every scenario.

In [ ]:
dss = scen.reference()
pub_runs = first.table[first.table["sample"] == 0]
t = pub_runs.merge(dss, on=["year", "sowing_shift"], suffixes=("", " DSSAT"))
rel = (t["HWAM"] - t["HWAM DSSAT"]).abs() / t["HWAM DSSAT"]
same_adat = int((t["ADAT"] == t["ADAT DSSAT"]).sum())
same_mdat = int((t["MDAT"] == t["MDAT DSSAT"]).sum())
say("check_yield", n=len(t), rel=rel.max())
say("check_dates", adat=same_adat, mdat=same_mdat, n=len(t))
say("dssat_per_scenario", sec=float(np.mean(scen.dssat_s)))

**Side by side with DSSAT on this machine.** `exp.dssat_batch` runs DSSAT on a set of perturbed cultivars of the 1982 season in one DSSAT batch run (one process; the time includes staging its run directory and writing its daily output files). Agri-JAX simulates the same written coefficients; the values must agree. The per-season times of both are then printed. The DSSAT time for the whole batch is an **extrapolation** from its measured per-season time, labelled as such; the two programs also do different amounts of work per season (DSSAT writes complete daily output files and runs modules the Agri-JAX day leaves out, Agri-JAX returns four values per season).

In [ ]:
n_dssat = 5 if SMALL else 50
db = exp.dssat_batch(treatment=4, cultivar={n: v[:n_dssat] for n, v in samples(2).items()})
ours = scen.run(db.cultivar).table
ours = ours[(ours["year"] == 1982) & (ours["sowing_shift"] == 0)].reset_index(drop=True)
rel = (ours["HWAM"] - db.table["HWAM"]).abs() / db.table["HWAM"]
say("batch_yield", n=db.seasons, rel=rel.max())
say("batch_dates", same=int((ours["MDAT"] == db.table["MDAT"]).sum()), n=db.seasons)

In [ ]:
n = second.timing["seasons"]
run_s = second.timing["run_s"]
per_dssat = db.elapsed_s / db.seasons
say("machine", machine=aj.machine())
say("dssat_speed", ms=1e3 * per_dssat, n=db.seasons, s=db.elapsed_s)
say("ours_speed", ms=1e3 * run_s / n, n=n, s=run_s)
say("compile", s=first.timing["compile_s"])
say("extrapolated", n=n, s=n * per_dssat)

## 4. Calibration

`aj.calibrate` fits the six `MZCER048.CUL` coefficients (P1, P2, P5, G2, G3, PHINT) of the cultivar to the observations of treatment 4 (silking and maturity dates, yield, tops weight and grain number at maturity, the LAI series), within the file's MINIMA / MAXIMA, by joint CMA-ES from several starts, with Agri-JAX alone; coefficients the observations carry no information on are fixed at the published value and reported. Treatment 6 is held out: evaluated, not fitted. The budget here is kept small for a free runtime (8 starts × 200 evaluations; a quick run uses less); the result is selected on the values a `.CUL` row can hold, and the published cultivar is kept when nothing beats it.

In [ ]:
starts, budget = (2, 48) if SMALL else (8, 200)
res = aj.calibrate(exp, treatments=[4], holdout=[6], starts=starts, budget=budget, seed=0)
print(res)

In [ ]:
aj.plot.calibration(res);

## 5. Round trip into DSSAT

The calibrated row is written into a copy of DSSAT's `MZCER048.CUL`, and `check_dssat()` runs the original DSSAT program with that row on both treatments: DSSAT's silking and maturity dates must equal Agri-JAX's, and its yield must be within 0.1 %.

In [ ]:
if os.path.exists("MZCER048_calibrated.CUL"):
    os.remove("MZCER048_calibrated.CUL")
print(res.write_cul("MZCER048_calibrated.CUL"))
res.check_dssat()

## Where to go next

* `exp.run` and `exp.scenarios` take other DSSAT maize experiments too (`aj.dssat.experiment("path/to/EXP.MZX")`, its weather files in the data's `example_data/Weather`); treatments that need a DSSAT process Agri-JAX does not implement yet (automatic irrigation, the CENTURY organic matter with surface residue, tillage, a water table …) are refused with the reason. `aj.calibrate` is limited to treatments whose nitrogen effect has been measured (the DSSAT v4.8.6 maize example treatments). Only the DSSAT maize example treatments are validated; results on other experiments should be checked against DSSAT the way this notebook does.
* The model runs nitrogen off: use treatments where nitrogen does not limit the crop (`exp.table()`).
* Source, validation and the process rules: [github.com/juksentang/agri-jax](https://github.com/juksentang/agri-jax). DSSAT-CSM is © the DSSAT Foundation and its contributors (BSD-3); the example data come from `DSSAT/dssat-csm-data` and `DSSAT/dssat-csm-os`.